# EDA: Planta de flotación de mineral de hierro
### Diagnóstico de calidad de datos y caracterización del proceso

**Autor:** Jesús Muñoz Sánchez - Ingeniero Industrial | Análisis de Datos \
**Contacto:** [LinkedIn](https://www.linkedin.com/in/jesus-munoz-sanchez/) · [GitHub](https://github.com/jesus3099jms3099/eda-planta-flotacion-hierro) \
**Herramientas:** Python (Pandas, NumPy, Matplotlib, Seaborn) \
**Dataset:** [Quality Prediction in a Mining Process — Kaggle](https://www.kaggle.com/datasets/edumagalhaes/quality-prediction-in-a-mining-process)

# 1. Contexto del dataset

Este proyecto es un análisis exploratorio (EDA) sobre datos operativos de una planta de flotación de mineral de hierro. La interrogante que se busca resolver es: ¿qué caracteriza el comportamiento del proceso y qué variables se asocian a la variabilidad del % de sílice en el concentrado final, la medida de calidad comercial del producto? El alcance es exploratorio: caracterización de las variables, diagnóstico de calidad de los datos y síntesis de hallazgos accionables.

El dataset proviene de una planta real de flotación inversa de mineral de hierro, publicado en Kaggle por Eduardo Magalhães Oliveira, quien lo anonimizó antes de publicarlo. Fue recopilado con un objetivo de control de calidad: la concentración de sílice en el producto final se mide en laboratorio solo una vez por hora, y el autor buscaba predecirla a partir de las variables de proceso para tomar acciones correctivas antes del resultado de laboratorio. En este proyecto el dataset se utiliza con fines exploratorios, no predictivos.

El periodo cubierto va de marzo a septiembre de 2017. El archivo tiene 24 columnas: una de fecha y hora (`date`) y 23 variables (4 resultados de laboratorio y 19 variables de proceso). Según la documentación, la frecuencia es de 20 segundos para las variables de proceso y de una hora para las de laboratorio.

Estructura de las variables:
- **Calidad de entrada (Feed):** `% Iron Feed` y `% Silica Feed` miden la calidad del mineral que ingresa.
- **Operativas primarias:** flujos de reactivos, flujo de pulpa, pH y densidad — el mayor impacto teórico sobre la calidad final.
- **Operativas secundarias:** niveles y flujos de aire en las celdas de flotación.
- **Calidad final (Concentrate):** `% Iron Concentrate` y `% Silica Concentrate`, esta última la variable de interés.

# 2. Perfilado de datos (data profiling)

**Objetivo:** evaluar la calidad estructural y de contenido del dataset identificando la presencia de valores nulos, registros duplicados, tipos de datos inconsistentes y cardinalidades anómalas antes de proceder a la manipulación de las variables.

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
# Cargar el dataset
ruta = "../data/MiningProcess_Flotation_Plant_Database.csv"
df_mining = pd.read_csv(ruta, decimal=",", parse_dates=["date"])

In [ ]:
# Tipos de datos y forma del dataset
df_mining.info()

El dataset tiene 737 453 filas y 24 columnas. La columna `date` es de tipo datetime y actúa como marca de tiempo; las demás variables (laboratorio y operativas) son de tipo float64. Aunque `.info()` no reporta valores nulos, a continuación se valida la calidad básica de los datos.

In [ ]:
# Calidad básica: nulos, ceros y rangos físicos
print(f"Nulos explícitos (NaN): {df_mining.isna().any().any()}")
print(f"Valores iguales a cero: {(df_mining == 0).any().any()}")
print("\nRango observado por variable (validación contra límites físicos):")
df_mining.agg(["min", "max"]).T

El dataset no presenta nulos explícitos (NaN) ni valores iguales a cero. Respecto a la validez física, las variables se encuentran en rangos coherentes: los porcentajes entre 0% y 100%, el pH entre 8.75 y 10.81 (consistente con el medio alcalino propio de la flotación), y los flujos, niveles y densidad son positivos en todo el rango. El único valor cercano a un límite es `Starch Flow` con 0.002 m³/h, un estado posible con dosificación nula.

In [ ]:
# Estructura temporal: continuidad y patrón de repetición del timestamp
t = df_mining["date"].drop_duplicates().sort_values()
print("Saltos entre timestamps consecutivos:")
print(t.diff().value_counts())

print("\nPatrón de repetición por timestamp (filas por hora):")
print(df_mining.groupby("date").size().value_counts())

Se identifican dos hechos estructurales:

1. **Existe un único gap temporal de 13 días y 7 horas**, que queda documentado como discontinuidad de la serie.
2. **Cada timestamp horario agrupa ~180 filas**: 4095 bloques de 180 elementos, uno de 174 y uno de 179, que suman el total del dataset. Si una hora contiene 180 registros, cada registro corresponde a un intervalo de 20 segundos (3600 s / 180 = 20 s). Esto revela que la marca de tiempo es horaria aunque las mediciones de proceso sean más frecuentes, los timestamps repetidos son estructurales, no errores, por lo que no deben eliminarse como duplicados de fecha.

In [ ]:
# Filas completas duplicadas (idénticas en las 24 columnas)
mascara_duplicados = df_mining.duplicated()
print(f"Filas duplicadas: {mascara_duplicados.sum()} "
      f"({100 * mascara_duplicados.mean():.2f}% del dataset)")
print("\nDías donde se concentran:")
print(df_mining.loc[mascara_duplicados, "date"].dt.date.value_counts().head())

Se encontraron 1171 filas completas duplicadas (0.16% del dataset), idénticas en las 23 variables y en el timestamp, concentradas en pocos días (principalmente 11 y 13 de julio). A diferencia de la repetición estructural de la marca horaria, estas sí son errores de registro y se eliminarán en la limpieza.

In [ ]:
# Comportamiento de refresco: valores distintos por bloque horario para cada variable
por_bloque = df_mining.groupby("date").nunique().agg(["mean", "std"]).T.round(1)
por_bloque.sort_values("mean")

Las variables se separan en tres grupos según su refresco dentro del bloque horario: el laboratorio de entrada (`% Iron/Silica Feed`) se mantiene constante dentro de cada hora; las variables operativas presentan cerca de 180 valores distintos por bloque (sí varían a ~20 s) y el laboratorio de salida (`% Iron/Silica Concentrate`) muestra un promedio de 10-14 valores por bloque con desviación estándar alta, lo que sugiere un comportamiento intermedio que requiere análisis individual.

In [ ]:
# Análisis del laboratorio de salida: distribución de valores únicos por hora
unicos_salida = df_mining.groupby("date")[["% Iron Concentrate", "% Silica Concentrate"]].nunique()

for col in unicos_salida:
    conteo = unicos_salida[col].value_counts().head(3)
    pct_conteo= (unicos_salida[col].value_counts(normalize=True) * 100).round(1)
    df_resumen = pd.DataFrame({'Cantidad': conteo, 'Porcentaje (%)': pct_conteo})
    print(df_resumen.head(), "\n")

# ¿Las horas con 180 valores (rampas) siguen algún patrón horario?
rampas = (unicos_salida == 180)
print("Rampas por hora del día (% Silica Concentrate):")
print(rampas.groupby(rampas.index.hour)["% Silica Concentrate"].sum().to_list())

Los resultados confirman una bimodalidad: cada bloque horario tiene 1 valor único (constante, una medición de laboratorio sostenida) o exactamente 180 valores (una rampa lineal). Para `% Iron Concentrate` existen 3881 horas constantes y 216 rampas (5.3%); para `% Silica Concentrate`, 3787 horas constantes y 310 rampas (7.6%). Las rampas son interpolación que rellena horas sin lectura de laboratorio, y se reparten en todas las horas del día sin patrón fijo, por lo que no corresponden a un evento programado. No son datos faltantes ni atípicos, sino valores sintéticos en lugar de medidos.

In [ ]:
# Evidencia visual: un día de laboratorio (valores constantes por hora vs rampas)
fecha_filtro = "2017-09-01"
dia = df_mining[df_mining["date"].dt.normalize() == fecha_filtro].reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle(f"Evolución de la sílice durante el {fecha_filtro}", fontsize=14)

ax1.plot(dia.index, dia["% Silica Feed"], color="C0")
ax1.set_ylabel("% Silica Feed")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(dia.index, dia["% Silica Concentrate"], color="C1")
ax2.set_ylabel("% Silica Concentrate")
ax2.set_xlabel("Tiempo (filas, 180 = 1 hora)")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(180))
ax2.set_xlim(0, len(dia))
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

El gráfico confirma el hallazgo: el laboratorio de entrada se sostiene constante cada hora (escalones), mientras el concentrado alterna entre horas constantes y tramos de rampa lineal (interpolación).

**Síntesis del perfilado de datos:**

El dataset comprende 737 453 registros y 24 columnas (una de tiempo, 4 de laboratorio y 19 de proceso), entre marzo y septiembre de 2017. El diagnóstico revela cinco hechos que condicionan el resto del análisis:

1. **La resolución temporal real es horaria, no de 20 segundos.** Cada hora agrupa 180 registros que comparten el mismo timestamp. La frecuencia de 20 s que sugiere la documentación es un artefacto de la marca de tiempo, no una medición real a esa cadencia.
2. **Existe un único gap de 13 días y 7 horas**, que parte la serie en dos tramos y debe respetarse como discontinuidad, no se interpola ni se promedia a través de él.
3. **Hay 1171 filas completas duplicadas (0.16%)**, concentradas en pocos días, las cuales probablemente sean errores de registro, distintos de la duplicación estructural de la marca horaria, se sugiere eliminar estos valores duplicados, ya que entorpecen la información.
4. **Las variables se separan en tres grupos según su actualización:** laboratorio de entrada y de salida a resolución horaria (un valor sostenido por hora), y variables operativas que sí varían a ~20 s.
5. **El concentrado contiene bloques interpolados:** 216 horas en `% Iron Concentrate` (5.3%) y 310 en `% Silica Concentrate` (7.6%) son rampas lineales que rellenan horas sin lectura de laboratorio. No son datos faltantes ni atípicos, sino valores sintéticos en lugar de medidos.

**Implicaciones para la limpieza:** eliminar las 1171 filas duplicadas; agregar a granularidad horaria, ya que esta es la cadencia del laboratorio y de la variable de interés; preservar el gap como discontinuidad; y marcar las horas interpoladas del concentrado para verificar después que ningún hallazgo dependa de ellas.

# 3. Limpieza de datos

**Objetivo:** corregir los problemas de calidad detectados en el perfilado, aplicando criterios técnicos para la remoción, agrupación y clasificación de datos, asegurando un dataset robusto y consistente para los análisis estadísticos posteriores.

In [ ]:
# Eliminar filas completas duplicadas
df_limpio = df_mining.drop_duplicates()
print(f"Duplicados eliminados: {len(df_mining) - len(df_limpio)}")

# Agregar a granularidad horaria (promedio por bloque)
df_horario = df_limpio.groupby("date").mean()
print(f"Filas tras agregación: {len(df_horario)} (una por hora)")

# Marcar las horas con % Silica Concentrate interpolado
nuq = df_mining.groupby("date")["% Silica Concentrate"].nunique()
df_horario["silica_interpolado"] = (nuq == 180)
n_interp = df_horario["silica_interpolado"].sum()
print(f"Horas con concentrado interpolado: {n_interp} "
      f"({100 * n_interp / len(df_horario):.1f}%)")

# Verificación de integridad
assert df_horario.index.is_unique
print("Integridad verificada: índice horario único.")

La limpieza reduce los 737 453 registros a 4097 observaciones horarias, se eliminaron las 1171 filas duplicadas y cada bloque horario se consolidó en un solo valor promedio. El `% Silica Concentrate` queda definido como su valor horario. El 7.6% de horas con concentrado interpolado queda señalado en `silica_interpolado`, lo que permitirá comprobar después que ningún hallazgo dependa de esos valores sintéticos. El gap de 13 días y 7 horas no se rellena: se conserva como discontinuidad para no fabricar una continuidad temporal que no existió.

# 4. Análisis univariado

**Objetivo:** describir cada variable de forma individual a través de su tendencia central, dispersión y forma de la distribución, con el fin de caracterizar el perfil de estabilidad de la planta y detectar comportamientos anómalos.

## Estadísticos descriptivos

In [ ]:
# Separar las variables de acuerdo a su naturaleza
lab_entrada = ["% Iron Feed", "% Silica Feed"]
lab_salida = ["% Iron Concentrate", "% Silica Concentrate"]
excluir = lab_entrada + lab_salida + ["silica_interpolado"]
operativas = [col for col in df_horario if col not in excluir]

# Coeficiente de variación porcentual
def cv_pct(x):
    return (x.std() / x.mean()) * 100

estadisticos = (df_horario.select_dtypes(include="number")
                .agg(["mean", "median", "std", cv_pct, "skew", "kurt"])
                .T.round(2))
estadisticos

**Interpretación de la tabla de estadísticos:**
- **Laboratorio de entrada:** el mineral que ingresa tiene comportamiento distinto según el elemento. El `% Iron Feed` es simétrico (skew 0.0), de variabilidad moderada (CV 9.2%) y centrado en ~56%. El `% Silica Feed` presenta asimetría positiva leve (skew 0.44) y es altamente variable (CV 46.5%). La variabilidad del mineral que ingresa proviene en mayor medida de la sílice.
- **Laboratorio de salida:** el `% Iron Concentrate` es muy estable (CV 1.7%, media ~65%, skew -0.52). Por el contrario, el `% Silica Concentrate` (variable de interés) tiene media 2.33%, mediana 2%, es altamente variable (CV 48.3%) y presenta asimetría positiva alta (skew 0.97). La planta controla la ley del hierro, mientras que la impureza (sílice) concentra la variabilidad y el riesgo de calidad.
- **Variables operativas:** existen variables altamente estables así como: Ore Pulp Flow (CV 2.1%), Air Flow de las columnas 04 y 05 (CV 0.8% y 1.1%), que sugieren control en setpoint; variables estables como el pH (CV 3.9%) y la densidad (CV 3.8%); y como grupo más variable, los niveles de las columnas (CV 17-26%) y los flujos de reactivos: Amina Flow (17.1%) y Starch Flow (33.1%). Los Air Flow de las columnas 01, 02, 03, 06 y 07 están sesgados negativamente: operan cerca de 300 m³/h con caídas ocasionales que arrastran el promedio.

In [ ]:
# Perfil de estabilidad de la planta: Coeficiente de Variación (CV) ordenado
cv = estadisticos["cv_pct"].sort_values()

plt.figure(figsize=(8, 10))
ax = cv.plot.barh()
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=4)
ax.set_xlim(0, cv.max() * 1.15)
plt.xlabel("Coeficiente de variación (%)")
plt.title("Variabilidad relativa por variable (CV)")
plt.tight_layout()
plt.savefig("../assets/coeficiente_variacion.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

Este gráfico ordenado ofrece una vista panorámica del perfil de estabilidad de la planta, con dos extremos marcados: en la parte superior, la inestabilidad está dominada por la impureza del mineral (`% Silica Feed` y `Concentrate`) y la dosificación de almidón (`Starch Flow`); en la base se agrupan las variables de baja variabilidad (flujos de aire, pulpa, pH y `% Iron Concentrate`).

**Nota:** el CV de las variables de entrada está levemente inflado por un periodo de valores congelados que se identifica y cuantifica más adelante; la lectura cualitativa del gráfico no cambia.

## Histogramas

In [ ]:
# Laboratorio de entrada
df_horario[lab_entrada].hist(bins=50, figsize=(12, 4))
plt.tight_layout()
plt.show()

**Laboratorio de entrada:** las dos variables difieren en forma y ambas revelan una anomalía que la tabla de estadísticos no mostraba. El `% Iron Feed` se distribuye de forma amplia y aproximadamente simétrica alrededor del 56%, pero presenta un pico aislado cerca del 64%. El `% Silica Feed` es sesgado a la derecha (entre ~8% y ~20%), también con un pico aislado alrededor del 6%. Se investigan ambos picos a continuación.

In [ ]:
# Investigación de los picos: valores congelados en el laboratorio de entrada
for var in lab_entrada:
    print(df_horario[var].value_counts().head(3))
    print("-" * 30)

# Moda
moda_iron = df_horario["% Iron Feed"].mode()[0]
moda_silica = df_horario["% Silica Feed"].mode()[0]

# Mascara de valores congelados
mask_iron_freeze = (df_horario["% Iron Feed"] == moda_iron)
mask_silica_freeze = (df_horario["% Silica Feed"] == moda_silica)

# Coincidencia y rango
coinciden = mask_iron_freeze.equals(mask_silica_freeze)
print(f"\nLos valores congelados coinciden en las misma posición: {coinciden}")
print(f"El periodo congelado representa el: {(mask_iron_freeze.sum() / len(df_horario)) * 100:.2f}% del dataset")
fechas_congeladas = df_horario[mask_iron_freeze].index
print(f"Rango: {fechas_congeladas.min()} a {fechas_congeladas.max()}")

# Estadísticos del laboratorio de entrada excluyendo el periodo congelado
print("\nEstadísticos sin el periodo congelado:")
df_horario.loc[~mask_iron_freeze, lab_entrada].agg(
    ["mean", "median", "std", cv_pct, "skew", "kurt"]).T.round(2)

**Hallazgo:** ambos picos corresponden a 792 horas (~20% del dataset) en las que el laboratorio de entrada quedó congelado en un único valor (64.03% hierro y 6.26% sílice), entre el 12-05-2017 y el 15-06-2017, el cual es un periodo sin muestreo de laboratorio cuya causa no es verificable al tratarse de datos anonimizados. Al excluir estas horas, el CV disminuye de 9.16% a 7.16% para el `% Iron Feed` y de 46.47% a 36.28% para el `% Silica Feed`, el cual a pesar de la corrección mantiene una variabilidad alta. El periodo queda marcado en `mask_iron_freeze` y `mask_silica_freeze` para las verificaciones de sensibilidad.

In [ ]:
# Variables operativas: reactivos, pulpa, pH y densidad
df_horario[operativas].filter(regex=r"^(?!.*(Air|Level))").hist(bins=50, figsize=(14, 8))
plt.tight_layout()
plt.show()

**Reactivos y pulpa:** `Starch Flow` presenta una distribución dispersa y bimodal (modas en ~2,200 y ~3,200), lo que sugiere que la planta opera bajo dos estrategias de dosificación. `Amina Flow` presenta sesgo a la izquierda. `Ore Pulp Flow` se mantiene casi estático en 400 m³/h, y el pH y la densidad muestran poca variación con un pico menor en el rango mínimo, lo que sugiere un mínimo recurrente.

In [ ]:
# Variables operativas: flujos de aire y niveles de las columnas
df_horario[operativas].filter(like="Air").hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

df_horario[operativas].filter(like="Level").hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

**Flujos de aire y niveles:** todos los flujos de aire son continuos pero están fuertemente concentrados en setpoints — una moda dominante cerca de 300 m³/h y modas secundarias en ~250 (y ~350 en las columnas 06 y 07), con baja densidad entre ellas correspondiente a las transiciones. Las columnas 04 y 05 son las más estrictas, en una banda muy estrecha en torno a 300. Su forma indica variables controladas en pocos estados, no fuentes de variación libre. Los niveles comparten un patrón notable: picos en posiciones regulares (350, 400, 450, 500…), lo que revela un control por escalones discretos coordinado entre columnas; son además el grupo operativo más variable (CV 17-26%) y sesgado a la derecha.

In [ ]:
# Laboratorio de salida
df_horario[lab_salida].hist(bins=50, figsize=(12, 4))
plt.tight_layout()
plt.show()

**Laboratorio de salida:** el `% Iron Concentrate` presenta una distribución unimodal con leve sesgo negativo (skew -0.52). El `% Silica Concentrate` presenta un marcado sesgo positivo (skew 0.97), donde la mayoría de horas concentradas en el rango inferior (1%-2%) y una cola que se extiende hacia valores superiores, los cuales son los episodios de alta impureza que constituyen el riesgo de calidad del producto.

## Detección de outliers

In [ ]:
# Contraste de dos métodos robustos: Tukey (IQR) y MAD
def outliers_tukey(x):
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    return ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()

def outliers_mad(x, umbral=3.5):
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0:
        return np.nan
    z = 0.6745 * (x - med) / mad
    return (z.abs() > umbral).sum()

columnas = df_horario.select_dtypes(include="number").columns
pd.DataFrame({
    "tukey": df_horario[columnas].apply(outliers_tukey),
    "mad": df_horario[columnas].apply(outliers_mad),
})

**Comentario:** al contrastar Tukey y MAD se observan discrepancias, el método MAD es excesivamente sensible en variables con alta concentración de datos en un solo valor (como los flujos de aire), marcando variabilidad operativa normal como anomalía.

**Decisión:** al tratarse de un EDA, no se eliminan outliers, ya que en un contexto industrial los valores extremos pueden representar estados operativos reales (arranques, paradas, perturbaciones) y no errores. Quedan documentado para distinguir ruido de sensores de estados críticos lo recomendable es aplicar detección de anomalías multivariada.

# 5. Análisis bivariado

**Objetivo:** identificar qué variables se asocian con la variable de interés (`% Silica Concentrate`). Se usa la correlación de Spearman por ser robusta ante distribuciones asimétricas y relaciones monótonas no lineales, características ya identificadas en el análisis univariado.

In [ ]:
# Correlación de Spearman de todas las variables contra la variable de interés
target = "% Silica Concentrate"
numericas = df_horario.select_dtypes(include="number")

corr_target = (numericas
               .corr(method="spearman")[target]
               .drop(target)
               .sort_values())

plt.figure(figsize=(8, 8))
ax = corr_target.plot.barh(color=np.where(corr_target < 0, "C3", "C0"))
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("Correlación de Spearman con % Silica Concentrate")
ax.set_title("Asociación de cada variable con la variable de interés")
plt.tight_layout()
plt.savefig("../assets/correlacion_spearman.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

**Interpretación:** la asociación más fuerte es la inversa entre `% Iron Concentrate` y `% Silica Concentrate` (ρ = -0.77), esperable por la naturaleza del proceso: hierro y sílice son fracciones complementarias del concentrado, por lo que esta relación describe la composición del producto. Entre las variables operativas, las de mayor asociación son: el `Amina Flow` con una relación directa (ρ = 0.22) y los niveles de las `Flotation Column 05 Level` y `Flotation Column 04 Level` con relaciones inversas (ρ = -0.21 y ρ = -0.20 respectivamente), mientras que las variables `Ore Pulp Flow` y el `Flotation Column 02 Level` muestran asociación débil a granularidad horaria.

In [ ]:
# Scatter de las 4 relaciones más fuertes con la variable de interés
top4 = corr_target.abs().sort_values(ascending=False).head(4).index

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, top4):
    ax.scatter(df_horario[col], df_horario[target], s=4, alpha=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel(target)
    ax.set_title(f"ρ = {corr_target[col]:.2f}")
fig.suptitle("Relaciones más fuertes con % Silica Concentrate", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación:** La relación entre `% Iron Concentrate` y `% Silica Concentrate` muestra una clara estructura lineal negativa y decreciente, donde se observa de forma evidente que a mayor porcentaje de hierro, menor es la presencia de sílice. En contraste, las variables operativas muestran nubes de puntos con alta dispersión y estructuras particulares, la variable `Amina Flow` exhibe una nube difusa con una leve tendencia positiva, mientras que las variables `Flotation Column 05 Level` y `Flotation Column 04 Level` se distribuyen en bandas o franjas verticales. Estas columnas verticales reflejan que la operación se concentra en ciertos valores de ajuste (setpoints) específicos (aproximadamente en 350, 400, 450 y 500), dentro de los cuales el porcentaje de sílice varía ampliamente.

In [ ]:
# Verificación de sensibilidad: ¿cambian las correlaciones al excluir
# las horas con concentrado interpolado y las horas con feed congelado?
mask_limpio = (~df_horario["silica_interpolado"]) & (~mask_iron_freeze)

corr_limpio = (df_horario.loc[mask_limpio]
               .select_dtypes(include="number")
               .corr(method="spearman")[target]
               .drop(target))

comparacion = pd.DataFrame({
    "todas_las_horas": corr_target,
    "sin_sinteticos": corr_limpio,
})
comparacion["diferencia"] = (comparacion.iloc[:, 0] - comparacion.iloc[:, 1]).abs()
comparacion.sort_values("diferencia", ascending=False).round(3).head(8)

**Comentario de sensibilidad:** al excluir las horas con valores sintéticos (310 horas interpoladas y 792 congeladas, que representan el 7.6% y ~20% del dataset respectivamente), las correlaciones se mantienen estables. Las variaciones máximas ocurren en las variables de alimentación `% Iron Feed` y `% Silica Feed` con cambios de ~0.11, mientras que las variables operativas críticas varían menos de 0.06. Esto confirma que los hallazgos bivariados son robustos y no dependen del ruido detectado en el perfilado.

**Recomendación:** se aconseja conservar estos registros en el dataset final en lugar de eliminarlos. Dado que el impacto en las   correlaciones es mínimo, descartar estas filas significaría perder valiosa información operativa real y continua de los sensores en planta. Mantenerlos es fundamental para preservar la variabilidad temporal del proceso de flotación, clave para el éxito de los posteriores modelos predictivos.

# 6. Análisis temporal

**Objetivo:** evaluar la estabilidad de la variable de interés a lo largo del periodo, identificando tendencias, cambios de régimen y el efecto del gap de 13 días documentado en el perfilado.

In [ ]:
# Serie temporal de % Silica Concentrate con media móvil de 24h
# respetando el gap como discontinuidad (no se traza línea a través de él)
serie = df_horario[target].copy()

saltos = serie.index.to_series().diff() > pd.Timedelta("1h")
segmento = saltos.cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
for _, tramo in serie.groupby(segmento):
    ax.plot(tramo.index, tramo.values, color="C0", lw=0.5, alpha=0.5)
    ax.plot(tramo.index, tramo.rolling(24, min_periods=12).mean(),
            color="C3", lw=1.5)

ax.set_ylabel("% Silica Concentrate")
ax.set_title("Evolución del % Silica Concentrate (azul: horario, rojo: media móvil 24h)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig("../assets/serie_tiempo.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

**Interpretación:** la media movil de 24 horas filtra el ruido de la serie y permite observar que el periodo sostenido con mayor concentración de sílice ocurre entre mediados de abril y mediados de mayo, rozando picos cercanos del 5%. Después del gap de marzo, la serie se reanuda manteniendo un nivel medio similar, pero mostrando ciclos de oscilación mejor definidos. La dispersión horaria (líneas azules) exhibe una alta volatilidad a lo largo de casi todo el registro, a excepción del mes de julio, donde el rango de variación diaria se reduce junto con el descenso generalizado en el porcentaje de la sílice.

In [ ]:
# Distribución mensual de la variable de interés
df_mes = df_horario[["% Silica Concentrate"]].copy()
df_mes["mes"] = df_mes.index.to_period("M").astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
df_mes.boxplot(column="% Silica Concentrate", by="mes", ax=ax, grid=False)
ax.set_xlabel("Mes")
ax.set_ylabel("% Silica Concentrate")
ax.set_title("Distribución mensual del % Silica Concentrate")
plt.suptitle("")
plt.tight_layout()
plt.show()

**Interpretación:** Los meses de marzo, abril y setiembre presentan las medianas más altas de `% Silica Concentrate`, lo que coinicide con los periodos de la serie de tiempo donde el proceso opera con niveles elevados de impureza. Por el contrario, entre junio y agosto, la mediana es baja, y el proceso experimenta una alta densidad de valores atípicos.

# 7. Síntesis ejecutiva

**Pregunta del análisis:** ¿qué caracteriza el comportamiento del proceso y qué variables se asocian a la variabilidad del % de sílice en el concentrado final?

**Hallazgos principales:**

1. **La resolución real del dataset es horaria.** La documentación sugiere mediciones cada 20 segundos, pero cada hora agrupa 180 registros con el mismo timestamp. Todo el análisis se realizó a granularidad horaria (4,097 observaciones después de la limpieza).

2. **El dataset contiene datos sintéticos no declarados.** Se identificaron 310 horas (7.6%) con `% Silica Concentrate` interpolado linealmente y 792 horas (~20%) con laboratorio de entrada congelado en un único valor. La verificación de sensibilidad confirmó que su impacto sobre las correlaciones es mínimo (variación máxima de ~0.11, concentrada en las variables de alimentación), por lo que los hallazgos son robustos y estos registros se conservan para preservar la continuidad temporal del proceso.

3. **La planta controla el hierro; la sílice concentra el riesgo de calidad.** El `% Iron Concentrate` es muy estable (CV 1.7%) mientras el `% Silica Concentrate` es altamente variable (CV 48.3%, asimetría positiva). La mayoría de horas opera con baja impureza, pero existen eventos de sílice elevada que constituyen el riesgo de calidad del producto.

4. **Las variables operativas más asociadas a la sílice del concentrado son el `Amina Flow` (ρ = 0.22, directa) y los niveles de las columnas de flotación 04 y 05 (ρ ≈ -0.20, inversa).** `% Iron Concentrate` (ρ = -0.77) es la relación inversa más fuerte, la cual es lógica por la naturaleza de la operación. Las asociaciones operativas, aunque suavizadas a granularidad horaria, son las manipulables; los scatter muestran que los niveles operan en setpoints discretos (350, 400, 450, 500) dentro de los cuales la sílice aún varía ampliamente, lo que sugiere que el control de nivel por sí solo no determina la calidad final.

5. **La calidad no es estacionaria en el tiempo.** El periodo de mayor concentración de sílice se sostiene entre mediados de abril y mediados de mayo (picos cercanos al 5%), y los meses de marzo, abril y septiembre presentan las medianas más altas. En contraste, julio muestra el menor nivel y la menor volatilidad diaria. Esto indica regímenes operativos distintos a lo largo del periodo y refuerza el valor de un monitoreo continuo frente a límites fijos.

**Limitaciones:** el dataset está anonimizado, sin metadatos de planta ni registro de eventos operativos, por lo que las causas de los datos congelados, las rampas interpoladas y el gap de 13 días no son verificables. Las correlaciones a granularidad horaria no implican causalidad ni capturan dinámicas más rápidas del proceso. La agregación horaria, necesaria por la resolución del laboratorio, suaviza la variabilidad operativa de 20 segundos.

**Recomendación de próximos pasos:** (1) construir un modelo predictivo de línea base del `% Silica Concentrate` a partir de las variables operativas, evaluando explícitamente el efecto de incluir o marcar las horas sintéticas identificadas aquí, (2) aplicar detección de anomalías multivariada para distinguir ruido de sensores de estados operativos críticos, (3) llevar los indicadores clave (nivel de sílice, ventanas de alta variabilidad y flujos de reactivos) a un dashboard de monitoreo en Power BI.